# Where Should LoRA Go? Probing *LoRA Without Regret* with Addresses

[LoRA Without Regret](https://thinkingmachines.ai/blog/lora/) (Schulman and Thinking Machines Lab, September 2025)
reports that where LoRA adapters sit matters more than their parameter count: on small and medium supervised datasets,
LoRA on the MLP layers matches full fine-tuning, adding attention adapters on top of them gains nothing, and LoRA on the
attention layers alone falls behind, even when given as many parameters as the MLP adapters.

This notebook probes that claim at small scale with Fine-Tuning Scheduler's experimental
[tunable views and addresses](https://finetuning-scheduler.readthedocs.io/en/latest/pytorch/tunable_views.html). It
is also the worked example of giving a model a view:

1. **one model, one view**: Qwen3-0.6B gets LoRA adapters on all seven of its projections, and a view that gives the
   adapters a role and every tensor an architecture-neutral component (`attn.q`, `mlp.down`, ...);
2. **the experimental conditions are addresses**: attention-only, MLP-only and all-layer LoRA are three one-line
   schedules (`role:adapter & component:attn.*`, ...) over that same model, so nothing but the address differs between
   them. Untrained adapters are exact no-ops (their `lora_B` starts at zero);
3. **a fourth condition only a scheduler can express**: MLP adapters first, then the attention adapters added halfway.

It is a probe, not a replication: a 0.6B model, a slice of GSM8K, a few hundred steps, one seed. Treat the result as
a hypothesis check, and scale it up (more steps, seeds, a learning-rate sweep) before drawing conclusions.

For one model and a schedule you write by hand, parameter names and regexes are usually all you need, and they stay
fully supported. Views and addresses earn their keep when conditions are compositions over a model's structure, as
here, or when one schedule must run on several models
([why Fine-Tuning Scheduler is evolving](https://finetuning-scheduler.readthedocs.io/en/latest/evolution.html)).

## Setup

The parameters below set the run's size (the docs render uses them as given; the tests shorten them).

In [ ]:
MODEL_ID = "Qwen/Qwen3-0.6B"
RANK = 16
ALPHA = 32
LR = 2e-4  # LoRA's best rate is about 10x full fine-tuning's (LoRA Without Regret); 2e-5 is a typical full rate here
MAX_STEPS = 300
EVAL_EVERY = 25
BATCH_SIZE = 8
SEQ_LEN = 256
N_TRAIN = 2400
N_EVAL = 256
SEED = 42

In [ ]:
import copy
import random
import warnings

import matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from finetuning_scheduler import TuningScheduler
from finetuning_scheduler.core.compile import compile_schedule
from finetuning_scheduler.testing import TunableViewConformance
from fts_examples.tunable_views.decoder_view import add_lora, DecoderView

warnings.filterwarnings("ignore", message=".*addresses.*experimental")  # said once per process; we know
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.bfloat16 if device.type == "cuda" else torch.float32
print(f"device: {device}, dtype: {dtype}")

## The Data

Grade-school math word problems with worked answers ([GSM8K](https://huggingface.co/datasets/openai/gsm8k)), as
`Question: ... Answer: ...` text. The loss is next-token cross-entropy over the whole sequence, and the evaluation set
is a fixed slice of the test split.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
gsm8k = load_dataset("openai/gsm8k", "main")


def encode(split, n):
    texts = [f"Question: {r['question']}\nAnswer: {r['answer']}{tokenizer.eos_token}" for r in gsm8k[split].select(range(n))]
    return tokenizer(texts, truncation=True, max_length=SEQ_LEN, padding="max_length", return_tensors="pt")


train_data, eval_data = encode("train", N_TRAIN), encode("test", N_EVAL)


def batches(data, size, shuffle_seed=None):
    order = list(range(data["input_ids"].shape[0]))
    if shuffle_seed is not None:
        random.Random(shuffle_seed).shuffle(order)
    for i in range(0, len(order) - size + 1, size):
        idx = order[i:i + size]
        ids, mask = data["input_ids"][idx].to(device), data["attention_mask"][idx].to(device)
        yield ids, mask, ids.masked_fill(mask == 0, -100)


print(f"{train_data['input_ids'].shape[0]} training and {eval_data['input_ids'].shape[0]} evaluation sequences of {SEQ_LEN} tokens")

## One Model, One View

The base model is loaded once and every condition starts from a copy of it, so every condition starts from identical
adapters too. `add_lora` wraps each target projection in a `LoRALinear`: the frozen base layer plus
$\frac{\alpha}{r} B A x$, with $B$ initialized to zero.

`DecoderView` keeps the model's own parameter names, so names and regexes still work through it, and adds two things a
schedule can address:

- **roles**: every LoRA tensor is an `adapter`, everything else `base`;
- **components**: each block tensor's architecture-neutral component, read from a small table for the model family
  (`fts_examples/tunable_views/decoder_view.py`). Layers need no table: the default structure finds the model's block
  list.

In [ ]:
torch.manual_seed(SEED)
base = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=dtype).to(device)
TARGETS = ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")
add_lora(base, TARGETS, r=RANK, alpha=ALPHA)
base.requires_grad_(False)

view = DecoderView(base, "qwen3")
structure = view.structure()
for name in ("model.layers.27.self_attn.q_proj.lora_A.weight", "model.layers.27.mlp.down_proj.base.weight",
             "model.layers.0.post_attention_layernorm.weight", "model.embed_tokens.weight"):
    print(f"{name:<52} role={view.roles().get(name, 'base'):<8} {structure.locate(name)}")

## Checking the View

`TunableViewConformance` runs every case of the protocol and every case Fine-Tuning Scheduler relies on: compiling a
schedule, thawing, an optimizer step and the refusals. It takes a factory, because the cases change the model.

In [ ]:
report = TunableViewConformance(lambda: DecoderView(copy.deepcopy(base), "qwen3")).run()
print("conformance:", "ok" if report.ok else report.failures)

## The Conditions Are Addresses

Each condition is a schedule. The first three have one phase, so they only select what trains. The fourth thaws the MLP
adapters first and adds the attention adapters halfway through.

| condition | phase 0 | phase 1 (from step `MAX_STEPS // 2`) |
| --- | --- | --- |
| attention only | `role:adapter & component:attn.*` | |
| MLP only | `role:adapter & component:mlp.*` | |
| all layers | `role:adapter` | |
| MLP, then attention | `role:adapter & component:mlp.*` | `role:adapter & component:attn.*` |

As regexes these would be `.*self_attn\.[qkvo]_proj\.lora_[AB].*` and the like: fine for this model, but tied to its
module names. The addresses mean the same tensors on any model whose view reports these components.

In [ ]:
CONDITIONS = {
    "attention only": {0: {"params": ["role:adapter & component:attn.*"]}},
    "MLP only": {0: {"params": ["role:adapter & component:mlp.*"]}},
    "all layers": {0: {"params": ["role:adapter"]}},
    "MLP, then attention": {0: {"params": ["role:adapter & component:mlp.*"], "max_transition_step": MAX_STEPS // 2},
                            1: {"params": ["role:adapter & component:attn.*"]}},
}
params = dict(base.named_parameters())
trainable = {}
for name, schedule in CONDITIONS.items():
    compiled = compile_schedule(schedule, view)
    names = [n for p in compiled.phases for n in p.physical]
    trainable[name] = sum(params[n].numel() for n in names)
    print(f"{name:<20} {len(names):>4} adapter tensors, {trainable[name] / 1e6:6.2f}M trainable parameters")

At equal rank, the attention-only and MLP-only conditions train nearly the same number of parameters on this model,
so their comparison is close to the matched-parameter comparison the paper emphasizes.

## Training

The loop is plain PyTorch. `TuningScheduler` thaws each phase's tensors, builds the optimizer over them, and adds the
next phase's tensors to it at the transition, which `step_end()` checks after each optimizer step.

In [ ]:
@torch.no_grad()
def evaluate(model):
    model.eval()
    losses = [model(input_ids=ids, attention_mask=mask, labels=labels).loss.item()
              for ids, mask, labels in batches(eval_data, BATCH_SIZE)]
    model.train()
    return sum(losses) / len(losses)


def train(schedule):
    model = copy.deepcopy(base)
    fts = TuningScheduler(model, schedule, view=DecoderView(model, "qwen3"),
                          optimizer=lambda groups: torch.optim.AdamW(groups, lr=LR, weight_decay=0.0), base_max_lr=LR)
    fts.setup()
    curve, step = [(0, evaluate(model))], 0
    for _ in fts.epochs(max_epochs=1):
        while step < MAX_STEPS:
            for ids, mask, labels in batches(train_data, BATCH_SIZE, shuffle_seed=SEED + step):
                model(input_ids=ids, attention_mask=mask, labels=labels).loss.backward()
                fts.optimizer.step()
                fts.optimizer.zero_grad()
                fts.step_end()
                step += 1
                if step % EVAL_EVERY == 0 or step == MAX_STEPS:
                    curve.append((step, evaluate(model)))
                if step == MAX_STEPS:
                    break
    del model, fts
    torch.cuda.empty_cache() if device.type == "cuda" else None
    return curve


results = {}
for name, schedule in CONDITIONS.items():
    results[name] = train(schedule)
    print(f"{name:<20} eval loss {results[name][0][1]:.4f} -> {results[name][-1][1]:.4f}")

## Results

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for name, curve in results.items():
    steps, losses = zip(*curve)
    ax.plot(steps, losses, marker="o", markersize=3, label=f"{name} ({trainable[name] / 1e6:.1f}M)")
ax.axvline(MAX_STEPS // 2, color="grey", linestyle=":", linewidth=1)
ax.set_xlabel("optimizer step")
ax.set_ylabel("GSM8K evaluation loss")
ax.set_title(f"LoRA placement on {MODEL_ID.split('/')[-1]} (rank {RANK})")
ax.legend()
plt.show()

best = min(results, key=lambda n: results[n][-1][1])
for name, curve in sorted(results.items(), key=lambda kv: kv[1][-1][1]):
    print(f"{name:<20} final eval loss {curve[-1][1]:.4f} ({curve[-1][1] - results[best][-1][1]:+.4f} vs best)")

## What This Shows

RESULTS_DISCUSSION_PLACEHOLDER

## Doing This for Your Own Model

The steps are the same for any model: inspect its parameter tree, decide which roles and components a schedule needs
to name, write the view (often only `CanonicalView` with `roles=`, plus a component table for architecture-neutral
components), check it with the conformance suite, then write the conditions as addresses.

- `fts_examples/tunable_views/decoder_view.py` holds this notebook's view, and `worked_example.py` beside it runs the
  same steps on two small, randomly initialized model families (Llama- and OPT-style) on CPU, with one address book and
  schedule for both.
- The `tunable-view-generator` agent skill, in the repository's `.agents/skills/`, packages these steps for agents in
  any harness that reads skills: a script that inspects a model and one that checks a view and the schedule compiled
  through it.